<a href="https://colab.research.google.com/github/anandtopu/100DaysOfMachineLearning/blob/master/Automating_ML_Retraining_with_Vertex_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Moving from infrastructure to MLOps is where a strong foundation in test automation truly shines. Building a continuous training (CT) pipeline in Vertex AI is essentially creating a massive, automated test suite where the "build artifact" is a machine learning model, and the "tests" validate data integrity and predictive accuracy before anything is allowed to reach production.

According to your design document, the system utilizes Vertex AI Pipelines, which is based on Kubeflow, to ensure reproducibility and continuous delivery.

Here is how to architect that automated retraining loop on Google Cloud Platform.

### 1. The Pipeline Architecture (The "What")

The pipeline is defined as a Directed Acyclic Graph (DAG) using the Kubeflow Pipelines (KFP) SDK. Each step runs in its own isolated container. Based on your project blueprint, the automated workflow follows these distinct steps:

1.
**Data Extraction:** Pulls the freshest labeled transaction data from BigQuery.


2.
**Data Validation:** Acts as the first automated test, using TensorFlow Data Validation (TFDV) to check for schema skew and drift. If the data is malformed, the pipeline fails early.


3.
**Transformation:** Normalizes numerical values and encodes categorical features.


4.
**Training:** Trains the primary Supervised XGBoost model, utilizing hyperparameter tuning.


5.
**Evaluation (The Quality Gate):** Computes the PR-AUC (Area Under Precision-Recall Curve) on a hold-out test set.


6.
**Conditional Registration:** The pipeline contains a logical gate; if the PR-AUC metric exceeds the required threshold, the artifact is uploaded to the Vertex Model Registry.



### 2. Python Implementation (The "How")

To build this, you will use the `kfp` library and Google's pre-built pipeline components. This script defines the pipeline logic and compiles it into a JSON file that Vertex AI can execute.

In [ ]:
import kfp
from kfp import dsl
from kfp.v2.dsl import component, Output, Model, Input, Dataset, Metrics
from google_cloud_pipeline_components.v1.bigquery import BigqueryQueryJobOp
from google_cloud_pipeline_components.v1.model import ModelUploadOp

PROJECT_ID = "fraud-detection-demo"
REGION = "us-central1"

# --- Define Custom Components ---

@component(packages_to_install=["tensorflow-data-validation", "pandas"])
def validate_data_op(
    dataset: Input[Dataset],
    validation_results: Output[Metrics]
):
    [cite_start]"""Uses TFDV to check for schema skew and drift[cite: 46]."""
    import tensorflow_data_validation as tfdv
    import pandas as pd

    # Logic to load data and generate statistics
    # If anomalies are found, raise an Exception to fail the pipeline (Automated testing)
    validation_results.log_metric("anomalies_found", 0)
    print("Data validation passed.")

@component(packages_to_install=["xgboost", "scikit-learn", "pandas"])
def train_and_evaluate_xgboost_op(
    dataset: Input[Dataset],
    model: Output[Model],
    metrics: Output[Metrics]
) -> float:
    [cite_start]"""Trains the XGBoost model and calculates PR-AUC[cite: 48, 49]."""
    import pandas as pd
    from sklearn.metrics import average_precision_score
    import xgboost as xgb

    # [cite_start]Placeholder for transformation and training logic [cite: 47, 48]
    # ...

    # [cite_start]Simulate evaluation [cite: 49]
    pr_auc_score = 0.88
    metrics.log_metric("pr_auc", pr_auc_score)

    # Save the model artifact
    model_path = model.path + ".bst"
    # booster.save_model(model_path)

    return pr_auc_score

# --- Define the Pipeline DAG ---

@dsl.pipeline(
    name="fraud-detection-retraining-pipeline",
    description="Automated ML pipeline for real-time fraud detection."
)
def fraud_pipeline(
    bq_query: str = f"SELECT * FROM `{PROJECT_ID}.fraud_detection_raw.transactions`",
    pr_auc_threshold: float = 0.85
):
    # [cite_start]Step 1: Data Extraction from BigQuery [cite: 45]
    extract_data_task = BigqueryQueryJobOp(
        project=PROJECT_ID,
        location=REGION,
        query=bq_query,
    )

    # [cite_start]Step 2: Validation (TFDV) [cite: 46]
    validate_task = validate_data_op(
        dataset=extract_data_task.outputs["destination_table"]
    )

    # [cite_start]Step 3 & 4 & 5: Transform, Train, and Evaluate [cite: 47, 48, 49]
    train_eval_task = train_and_evaluate_xgboost_op(
        dataset=extract_data_task.outputs["destination_table"]
    ).after(validate_task)

    # [cite_start]Step 6: Conditional Registration Gate [cite: 49, 50]
    # Only upload to the registry if the model passes the PR-AUC test
    with dsl.Condition(
        train_eval_task.output >= pr_auc_threshold,
        name="quality-gate-passed"
    ):
        ModelUploadOp(
            project=PROJECT_ID,
            location=REGION,
            display_name="xgboost-fraud-model",
            unmanaged_container_model=train_eval_task.outputs["model"]
        )

# --- Compile the Pipeline ---
if __name__ == "__main__":
    kfp.v2.compiler.Compiler().compile(
        pipeline_func=fraud_pipeline,
        package_path="fraud_pipeline.json"
    )
    print("Pipeline compiled successfully to fraud_pipeline.json")

### 3. Executing the Pipeline

Once compiled into the `fraud_pipeline.json` file, you can submit it to Vertex AI manually via the Google Cloud Console, or automate it fully by triggering the pipeline via Cloud Functions or Cloud Scheduler on a nightly or weekly cadence.

This establishes a highly controlled environment. New data flows in, automated tests validate it, the model trains, and it is mathematically verified against your baseline threshold before it ever touches the model registry.

Would you like to map out the **Vertex Model Monitoring** setup next to implement the automated drift detection and alerting  for the deployed endpoints?